# 02 - Reader Feasibility Test

This notebook checks whether Qwen3.5-4B can read numbers off a financial table before the pipeline is built around it. If the model cannot do this, the page-image half of the project needs a different model, and it is better to learn that now than after building everything.

- For the test, 3 questions were chosen from the dev split whose whose evidence page is a statement full of numbers.
- Each page is handed to the model twice, once as an image and once as extracted text, and the answers are compared to the reference answers manually.

This notebook runs on Google Colab, because the model needs a GPU.

In [1]:
!pip install -q -U transformers accelerate pymupdf
!git clone https://github.com/dogamentese/financebench-rag.git
%cd financebench-rag
!python scripts/download_data.py

Cloning into 'financebench-rag'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 607.14 KiB | 2.04 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/financebench-rag
[1/84] 3M_2018_10K
[2/84] 3M_2022_10K
[3/84] 3M_2023Q2_10Q
[4/84] ACTIVISIONBLIZZARD_2019_10K
[5/84] ADOBE_2015_10K
[6/84] ADOBE_2016_10K
[7/84] ADOBE_2017_10K
[8/84] ADOBE_2022_10K
[9/84] AES_2022_10K
[10/84] AMAZON_2017_10K
[11/84] AMAZON_2019_10K
[12/84] AMCOR_2020_10K
[13/84] AMCOR_2022_8K_dated-2022-07-01
[14/84] AMCOR_2023Q2_10Q
[15/84] AMCOR_2023Q4_EARNINGS
[16/84] AMCOR_2023_10K
[17/84] AMD_2015_10K
[18/84] AMD_2022_10K
[19/84] AMERICANEXPRESS_2022_10K
[20/84] AMERICANWATERWORKS_2020_10K
[21/84] AMERICANWATERWORKS_2021_10K
[22/84] AMERICANWATERWORKS_2022_10K
[23/84] BESTBUY_2017_10K
[24/84] BESTBUY_2019_10K
[25/84] BE

## 1. Load the model

Qwen3.5-4B is an open-weights vision language model, meaning it accepts both text and images as input. The first run downloads about 9 GB of weights from Hugging Face, which takes a few minutes. The model thinks step by step by default, which is slow on a free GPU, so thinking is turned off when asking the questions below.

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_id = "Qwen/Qwen3.5-4B"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

GPU memory after loading: 9.1 GB


## 2. Prepare the 3 test cases

The 3 questions below come from the dev split in `data/split.json`. All three are metrics questions whose evidence page is a financial statement, so answering them means reading numbers out of a table.

For each question, the evidence page is rendered as an image at 150 dpi and extracted as text with PyMuPDF, the same two input modes the project compares.

In [3]:
import json
import time
from pathlib import Path

import pymupdf

FB = Path.cwd() / "data" / "financebench"
TEST_IDS = ["financebench_id_02981", "financebench_id_04672", "financebench_id_04980"]

questions = {q["financebench_id"]: q for q in (json.loads(line) for line in open(FB / "financebench_open_source.jsonl"))}

cases = []
for fid in TEST_IDS:
    q = questions[fid]
    page_num = q["evidence"][0]["evidence_page_num"]
    page = pymupdf.open(FB / "pdfs" / f"{q['doc_name']}.pdf")[page_num]
    img_path = Path(f"/content/{fid}.png")
    page.get_pixmap(dpi=150).save(img_path)
    cases.append({
        "id": fid,
        "question": q["question"],
        "answer": q["answer"],
        "text": page.get_text(),
        "image": img_path,
    })
    print(fid, "|", q["doc_name"], "| page", page_num)

financebench_id_02981 | CORNING_2021_10K | page 64
financebench_id_04672 | 3M_2018_10K | page 57
financebench_id_04980 | PEPSICO_2021_10K | page 62


## 3. Ask each question in both modes

The same question is asked twice, once with the page as an image and once with the page as text. Generation is deterministic (`do_sample=False`), so re-running gives the same answers. Each answer is printed next to the reference answer, with the time it took.

In [4]:
PROMPT = """{question}

Work through the question step by step and show any calculation.
End with one line in the form: Final answer: <value>"""


def ask(question, *, image=None, text=None):
    content = []
    if image is not None:
        content.append({"type": "image", "path": str(image)})
    if text is not None:
        content.append({"type": "text", "text": f"Here is a page from a financial report:\n\n{text}"})
    content.append({"type": "text", "text": PROMPT.format(question=question)})
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)
    start = time.time()
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    response = processor.decode(new_tokens, skip_special_tokens=True).strip()
    return response, len(new_tokens), time.time() - start


def final_answer(response):
    for line in reversed(response.splitlines()):
        if "final answer:" in line.lower():
            return line.split(":", 1)[1].strip(" *")
    return "(no final answer line)"


results = []
for case in cases:
    print("=" * 80)
    print("Q:", case["question"])
    print("reference answer:", case["answer"])
    for mode in ["image", "text"]:
        response, n_tokens, seconds = ask(case["question"], **{mode: case[mode]})
        results.append({"id": case["id"], "mode": mode, "response": response})
        print(f"[{mode}, {seconds:.0f}s, {n_tokens} tokens] {final_answer(response)}")
    print()

print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

Q: Taking into account the information outlined in the income statement, what is the FY2019 - FY2021 3 year average unadjusted operating income % margin for Corning? Answer in units of percents and round to one decimal place.
reference answer: 10.3%


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[image, 35s, 410 tokens] 10.3
[text, 38s, 512 tokens] (no final answer line)

Q: Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
reference answer: $8.70
[image, 14s, 163 tokens] 8.738
[text, 26s, 352 tokens] 8.738

Q: What is the FY2021 capital expenditure amount (in USD billions) for PepsiCo? Respond to the question by assuming the perspective of an investment analyst who can only use the details shown within the statement of cash flows.
reference answer: $4.60
[image, 20s, 238 tokens] 4.625
[text, 26s, 348 tokens] 4.625

peak GPU memory: 9.8 GB


## 4. The working on the calculation question

The cell below prints both full responses for the Corning question, to check whether the model averages all three years. The 3 margins on the page are 15.0% (2021), 4.5% (2020) and 11.35% (2019), which average to 10.3%.

In [5]:
for r in results:
    if r["id"] == "financebench_id_02981":
        print(f"----- {r['mode']} -----")
        print(r["response"])
        print()

----- image -----
To calculate the **3-year average unadjusted operating income % margin** for Corning from FY2019 to FY2021, we need to:

1. Extract **Operating Income** and **Net Sales** for each year (2019, 2020, 2021).
2. Calculate **Operating Income Margin** for each year:  
   → Operating Income Margin = (Operating Income / Net Sales) × 100%
3. Compute the **average** of these three margins.

---

From the Consolidated Statements of Income:

**FY2021:**
- Net Sales = $14,082 million
- Operating Income = $2,112 million  
→ Margin = (2,112 / 14,082) × 100 ≈ 14.997% → **15.0%** (rounded to one decimal)

**FY2020:**
- Net Sales = $11,303 million
- Operating Income = $509 million  
→ Margin = (509 / 11,303) × 100 ≈ 4.503% → **4.5%**

**FY2019:**
- Net Sales = $11,503 million
- Operating Income = $1,306 million  
→ Margin = (1,306 / 11,503) × 100 ≈ 11.353% → **11.4%**

---

Now, compute the **average** of the three margins:

(15.0 + 4.5 + 11.4) / 3 = 30.9 / 3 = **10.3%**

---

Final an

## Verdict

First question needs a small calculation, and the next two are direct lookups.

- image mode: 3 of 3 correct, about 14 to 35 s per answer
- text mode: 2 of 3 correct, about 26 to 38 s per answer; the Corning working reached 10.3 but hit the token limit before the final answer line

Both direct lookups were read correctly in both modes.
For the pipeline, raise max_new_tokens above 1024 so multi-step answers are not cut off.